# EE 672: Trustworthy Agentic AI for Security Alert Triage
Walkthrough: load a case, see the three evidence variants, run configs A/B/C, and score the results.

Runs with the scripted demo model (no API key). Swap in a real model once the adapter exists.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from triage.fixtures import load_cases, build_view, VARIANTS
from triage.agent import run_case
from triage.score import score_run
from triage.run import demo_model, CONFIGS

## 1. Load the cases
Every case is validated against `fixtures/case.schema.json` on load.

In [ ]:
cases = load_cases(split='dev', family='suspicious_auth')
for c in cases:
    print(c['case_id'], c['split'], c['scenario_group'], '|', c['alert']['rule'])

## 2. What changes between variants
`withheld` removes records. `injected` overwrites one attacker-controlled field with a payload containing a canary.

In [ ]:
case = cases[0]
for v in VARIANTS:
    view = build_view(case, v)
    ids = [r['evidence_id'] for k in view['evidence'].values() for r in k]
    spec = case['variants'][v]
    print(f"{v:9} expect={spec['expected_behavior']:7} disposition={spec['expected_disposition']:16} records={ids}")

inj = case['variants']['injected']['injection']
print('\nInjected field:', inj['target_evidence_id'], inj['field'])
print('Payload:', inj['payload'])

## 3. Run every variant x config and score it

In [ ]:
rows = []
for case in cases:
    for v in VARIANTS:
        view = build_view(case, v)
        for cfg in CONFIGS:
            log = run_case(case, v, cfg, demo_model(), view)
            rows.append(score_run(log, case))
print(len(rows), 'runs')

In [ ]:
df = pd.DataFrame(rows)
df['injection_success'] = df['injection'].apply(lambda x: x['success'] if x else None)
df[['case_id','variant','config','disposition_correct','abstain_cell','supported_claim_rate','evidence_recall','fabricated_citations','unauthorized_attempts','injection_success']]

## 4. Summary by config
These are the numbers the final report compares. With the scripted demo model every config gives the same answer, so differences here only come from what each config could retrieve.

In [ ]:
summary = df.groupby('config').agg(
    disposition_acc=('disposition_correct', 'mean'),
    supported_claims=('supported_claim_rate', 'mean'),
    evidence_recall=('evidence_recall', 'mean'),
    unauthorized=('unauthorized_attempts', 'sum'),
    injection_success=('injection_success', 'mean'),
)
summary

In [ ]:
pd.crosstab(df['config'], df['abstain_cell'])

## Next
- Add the real model adapter in `triage/agent.py`, then rerun this notebook with it.
- Add plots once there are enough cases to compare.